# Article Recommender System : Build It From Scratch

In this challenge you will build a simple article recommender system using only NumPy.

By the end you will have a working system that, given an article a user just read,
finds and recommends the most similar articles from a collection.

---

## How does it work?

1. Each article is described by a vector of numbers —> one score per topic
2. We measure how similar two articles are by comparing their vectors
3. We recommend the articles with the highest similarity score

The similarity measure we use is called **cosine similarity**:

$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

A result of **1.0** means identical direction. A result of **0.0** means completely unrelated.

---

## Scoring

| Task | Description | Points |
|---|---|---|
| Task 1 | Load and explore the data | 10 |
| Task 2 | Normalize the vectors | 15 |
| Task 3 | Compute the similarity matrix | 20 |
| Task 4 | Find the most similar articles | 20 |
| Task 5 | Build the recommender function | 20 |
| Task 6 | Test and evaluate | 15 |
| **Total** | | **100** |

---

## Rules

- You may only use **NumPy**. No scikit-learn, no pandas.
- Read each task fully before writing code.
- Hints are available, try without them first.
- Run all cells from top to bottom.

---
## Setup

Run this cell first. It imports NumPy and loads the data from the CSV file.

In [1]:
import numpy as np

# Load the CSV file
# we only use NumPy
raw = np.genfromtxt('articles_data.csv', delimiter=',', dtype=str, encoding='utf-8')

# Row 0 is the header —> skip it
# Column 0 is the index, column 1 is the title, columns 2-7 are the topic scores
article_titles  = list(raw[1:, 1])                        # column 1 = titles
article_vectors = raw[1:, 2:].astype(np.float32)          # columns 2 to 7 = vectors
topic_names     = list(raw[0, 2:])                        # header row, columns 2 to 7

print('Topics:          ', topic_names)
print('Number of articles:', len(article_titles))
print('Vector shape:      ', article_vectors.shape)
print()
print('Articles:')
for i, title in enumerate(article_titles):
    print(f'  [{i}] {title}')
    print(f'       {article_vectors[i]}')

Topics:           [np.str_('ai'), np.str_('data'), np.str_('health'), np.str_('climate'), np.str_('finance'), np.str_('sport')]
Number of articles: 8
Vector shape:       (8, 6)

Articles:
  [0] Introduction to Machine Learning
       [0.9 0.8 0.  0.  0.1 0. ]
  [1] How Neural Networks Work
       [0.8 0.7 0.  0.1 0.  0. ]
  [2] Climate Change and Renewable Energy
       [0.  0.1 0.2 0.9 0.1 0. ]
  [3] The Future of Electric Vehicles
       [0.1 0.2 0.  0.7 0.3 0. ]
  [4] Healthy Eating on a Budget
       [0.  0.  0.9 0.1 0.3 0. ]
  [5] Exercise and Mental Health
       [0.  0.  0.8 0.  0.  0.5]
  [6] Stock Market Basics
       [0.1 0.2 0.  0.  0.9 0.1]
  [7] Data Science for Finance
       [0.5 0.8 0.  0.  0.7 0. ]


---
## Task 1 : Explore the data (10 points)

The data is already loaded. Your job is to inspect it using NumPy.
Do not hardcode any answers —> compute them from `article_vectors` and `article_titles`.

**1a (3 points):** Print the number of articles and the number of topics.

In [3]:
# Task 1a — your code here

n_articles = article_vectors.shape[0]
n_topics   = article_vectors.shape[1]    

print('Number of articles:', n_articles)
print('Number of topics:  ', n_topics)

Number of articles: 8
Number of topics:   6


**1b (3 points):** Print the vector for article at index 3 only.

In [4]:
# Task 1b : your code here
index = 3
vector = article_vectors[index]
print('Title: ', article_titles[index])
print('Vector:', vector)

Title:  The Future of Electric Vehicles
Vector: [0.1 0.2 0.  0.7 0.3 0. ]


**1c (4 points):** Find and print the title of the article that has the highest score
in the `finance` topic (column index 4).

In [5]:
# Task 1c : your code here

finance_col   = article_vectors[:,4]          # extract the finance column
top_article   = np.argmax(finance_col)         # index of the article with the highest finance score

print('Most finance-related article:', article_titles[top_article])
print('Finance score:               ', finance_col[top_article])

Most finance-related article: Stock Market Basics
Finance score:                0.9


<details>
<summary>Hint for 1a (click to expand)</summary>

`article_vectors.shape` returns a tuple `(rows, columns)`.
You can unpack it: `n_articles, n_topics = article_vectors.shape`

</details>

<details>
<summary>Hint for 1c (click to expand)</summary>

Use `article_vectors[:, 4]` to get the finance column for all articles.
Then use `np.argmax(...)` to find the index of the highest value.

</details>

---
## Task 2 : Normalize the vectors (15 points)

Before computing similarity, we need to **normalize** each vector so its length equals 1.0.
A vector with length 1 is called a **unit vector**.

To normalize, divide each vector by its own length (its L2 norm):

$$\hat{v} = \frac{v}{\|v\|}$$

**Why?** Without normalization, an article with larger numbers would appear more similar
to everything — not because it is more related, but just because its values are bigger.
Normalization removes that bias.

**2a (5 points):** Compute the L2 norm of each row. Store it in `norms`.
Print the shape of `norms` (it should be `(8,)`).

In [7]:
# Task 2a : compute the norm of each row

norms = np.linalg.norm(article_vectors, ord=2, axis=1)

print('Norms shape:', norms.shape)   # should be (8,)
print('Norms:      ', norms.round(4))

Norms shape: (8,)
Norms:       [1.2083 1.0677 0.9327 0.7937 0.9539 0.9434 0.9327 1.1747]


**2b (10 points):** Divide each row of `article_vectors` by its norm.
Store the result in `normalized_vectors`.
Verify that every row now has norm = 1.0.

In [13]:
# Task 2b : normalize each row

norms_modified = norms.reshape(-1, 1)  
normalized_vectors = article_vectors / norms_modified

# Verify — all values should be 1.0
print('Norms after normalization:')
print(np.linalg.norm(normalized_vectors, axis=1).round(6))

Norms after normalization:
[1. 1. 1. 1. 1. 1. 1. 1.]


<details>
<summary>Hint for 2a (click to expand)</summary>

Use `np.linalg.norm(article_vectors, axis=1)`.
The `axis=1` means compute one norm per row.

</details>

<details>
<summary>Hint for 2b (click to expand)</summary>

`norms` has shape `(8,)` and `article_vectors` has shape `(8, 6)`.
You cannot divide them directly, the shapes do not align.
Reshape `norms` to `(8, 1)` first: `norms.reshape(-1, 1)`
This tells NumPy to divide each row by its own norm.

</details>

---
## Task 3 : Compute the similarity matrix (20 points)

Now compute the **cosine similarity between every pair of articles** at once.

Because our vectors are already normalized, cosine similarity is just the dot product.
We can compute all pairs in one line using matrix multiplication:

$$\text{Similarity Matrix} = \hat{X} \times \hat{X}^T$$

The result is an `(8, 8)` matrix where entry `[i, j]` is the similarity
between article `i` and article `j`.

**3a (10 points):** Compute the similarity matrix and store it in `similarity_matrix`.
Print its shape — should be `(8, 8)`.

In [ ]:
# Task 3a : compute the similarity matrix

similarity_matrix = ___

print('Shape:', similarity_matrix.shape)   # should be (8, 8)

**3b (10 points):** Print the full matrix rounded to 2 decimal places.
Then print the diagonal values.
What do you expect the diagonal to contain, and why?

In [ ]:
# Task 3b : print the matrix and the diagonal

print('Similarity matrix:')
print(___)

print('\nDiagonal (similarity of each article with itself):')
print(___)

<details>
<summary>Hint for 3a (click to expand)</summary>

Use the `@` operator and `.T` for transpose:
`normalized_vectors @ normalized_vectors.T`

</details>

<details>
<summary>Hint for 3b (click to expand)</summary>

Use `.round(2)` on the matrix to round all values.
Use `np.diag(similarity_matrix)` to extract the diagonal.
The diagonal should be all 1.0 — every article is perfectly similar to itself.

</details>

---
## Task 4 : Find the most similar articles (20 points)

Given one article, find the other articles most similar to it.

**4a (5 points):** Extract the similarity scores for article index 0.
Print the scores.

In [ ]:
# Task 4a

article_index = 0
scores = ___

print('Scores for article 0:')
for i, s in enumerate(scores):
    print(f'  [{i}] {article_titles[i]}: {s:.4f}')

**4b (10 points):** Sort the articles by similarity score from highest to lowest.
Remove article 0 itself from the list.
Store the sorted indices in `sorted_indices`.

In [ ]:
# Task 4b

# Sort from highest to lowest
sorted_indices = ___

# Remove the article itself
sorted_indices = ___

print('Articles sorted by similarity to article 0:')
for idx in sorted_indices:
    print(f'  [{idx}] {article_titles[idx]}: {scores[idx]:.4f}')

**4c (5 points):** Print the top 3 recommendations with their titles and scores.

In [ ]:
# Task 4c

print(f'Because you read: "{article_titles[article_index]}"')
print('Top 3 recommendations:')
for rank, idx in enumerate(___, start=1):
    print(f'  {rank}. {article_titles[idx]}  (similarity: {scores[idx]:.4f})')

<details>
<summary>Hint for 4a (click to expand)</summary>

`similarity_matrix[article_index]` gives you the full row for that article —
one score per article in the collection.

</details>

<details>
<summary>Hint for 4b : sorting (click to expand)</summary>

`np.argsort(scores)` sorts from lowest to highest.
Add `[::-1]` to reverse: `np.argsort(scores)[::-1]`

</details>

<details>
<summary>Hint for 4b : removing self (click to expand)</summary>

Use a list comprehension to filter out the article itself:
`[i for i in sorted_indices if i != article_index]`

</details>

<details>
<summary>Hint for 4c (click to expand)</summary>

Use `sorted_indices[:3]` to get the first 3 items from the sorted list.

</details>

---
## Task 5 : Build the recommender function (20 points)

Now wrap everything from Task 4 into a single reusable function.

Write a function called `recommend` with these parameters:

| Parameter | Type | Description |
|---|---|---|
| `article_index` | int | Index of the article the user just read |
| `similarity_matrix` | 2D array | The matrix from Task 3 |
| `titles` | list | The list of article titles |
| `top_n` | int | How many recommendations to return (default: 3) |

It must **return** a list of `(title, score)` tuples — not just print them.

In [ ]:
# Task 5 : write the full function

def recommend(article_index, similarity_matrix, titles, top_n=3):
    """
    Returns the top_n most similar articles to the one at article_index.
    Returns a list of (title, score) tuples.
    """
    # your code here
    pass


# Test it — should print 3 recommendations for article 0
results = recommend(0, similarity_matrix, article_titles)
print('Test — recommendations for article 0:')
for title, score in results:
    print(f'  - {title}  ({score:.4f})')

<details>
<summary>Hint — structure of the function (click to expand)</summary>

Your function should do the same steps as Task 4 — just use the parameters
instead of hardcoded values.

At the end, build and return the list like this:
```python
results = []
for idx in sorted_indices[:top_n]:
    results.append((titles[idx], scores[idx]))
return results
```

</details>

---
## Task 6 : Test and evaluate (15 points)

**6a (5 points):** Run `recommend` for every article and print all results.
Format the output clearly.

In [ ]:
# Task 6a

for i in range(___):
    print(f'\nBecause you read: "{article_titles[i]}"')
    results = recommend(___, ___, ___)
    for title, score in results:
        print(f'  - {title}  ({score:.4f})')
    print('-' * 50)

**6b (5 points):** A user just read article 7 ("Data Science for Finance").
Get the **top 2** recommendations for them and print them.

In [ ]:
# Task 6b

results = recommend(___, ___, ___, top_n=___)
print(f'Because you read: "{article_titles[7]}"')
print('Top 2 recommendations:')
for title, score in results:
    print(f'  - {title}  ({score:.4f})')

**6c (5 points):** Answer the two questions below in the text cell.

Look at the recommendations for article 2 ("Climate Change") and article 5 ("Exercise and Mental Health").

1. Do the recommendations make sense? Explain why or why not.
2. What is one thing this recommender cannot do that a real-world recommender can?

*Write your answer here.*

---
## What you built

```
articles_data.csv
       |
       v
Load with np.genfromtxt             (Setup)
       |
       v
Explore: shape, slicing, argmax     (Task 1)
       |
       v
Normalize each vector               (Task 2)
       |
       v
Similarity matrix = X @ X.T         (Task 3)
       |
       v
Sort by score, remove self          (Task 4)
       |
       v
Reusable recommend() function       (Task 5)
       |
       v
Test and evaluate                   (Task 6)
```

| NumPy operation | Where you used it |
|---|---|
| `np.genfromtxt` | Loading the CSV file |
| `array.shape` | Checking dimensions |
| `array[:, col]` | Selecting a column |
| `np.argmax` | Finding the highest value |
| `np.linalg.norm` | Computing vector lengths |
| Broadcasting | Dividing all rows at once |
| `@` and `.T` | Similarity matrix in one line |
| `np.argsort` | Ranking by score |
| `np.diag` | Reading the diagonal |
| Slicing `[:3]` | Taking the top N results |